# LAB-D1-04: Build a Network That Thinks Forward

**Purpose:** Assemble a vectorized forward-only NumPy network and explain correct and incorrect outputs using raw geometry, a decision region, hidden activations, and a trace.

**Objectives:** `OBJ-D1-06`, `OBJ-D1-07`, retrieval of `OBJ-D1-04` and `OBJ-D1-05`  
**Estimated duration:** 60 minutes live; under 20 seconds compute  
**Prerequisites:** `LESSON-D1-06`, `LAB-D1-02`, `LAB-D1-03`; batch-first dense operations and activation invariants  
**Environment:** CPU only; NumPy, matplotlib, scikit-learn; seeded generated data; no network or download

This is inference only: no training or parameter updates occur. Workflow: **Observe -> Predict -> Modify -> Run -> Visualize -> Diagnose -> Explain -> Extend**. Restart and run in order.

In [ ]:
import copy
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

SEED = 41
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__} | scikit-learn {sklearn.__version__}")
print("Runtime target: local/Colab CPU; no network or GPU required.")

## Observe: Seeded Moons and a Fixed Evidence Split

The data contains two noisy interleaving arcs. The fixed stratified split records which examples were available during the offline parameter-fitting process and which examples are used for held-out investigation here. This participant notebook never fits a model.

Class is encoded by marker and color. Six held-out examples receive neutral names so you can track them across every panel.

In [ ]:
X_all, y_all = make_moons(n_samples=300, noise=0.27, random_state=SEED)
all_indices = np.arange(len(X_all))
train_indices, evidence_indices = train_test_split(
    all_indices, test_size=0.30, random_state=SEED, stratify=y_all
)
X_train, y_train = X_all[train_indices], y_all[train_indices]
X_evidence, y_evidence = X_all[evidence_indices], y_all[evidence_indices]

probe_indices = np.array([4, 127, 265, 228, 273, 181])
probe_names = np.array(["west-window", "lower-gate", "upper-arc", "center-step", "east-rim", "east-notch"])
assert set(probe_indices).issubset(set(evidence_indices))
X_probes = X_all[probe_indices]
y_probes = y_all[probe_indices]

assert X_train.shape == (210, 2) and X_evidence.shape == (90, 2)
assert X_probes.shape == (6, 2) and y_probes.shape == (6,)
print("Data ready: 210 offline-fit examples, 90 held-out evidence examples, 6 named probes.")

In [ ]:
CLASS_STYLES = [(0, "o", "class 0"), (1, "^", "class 1")]


def scatter_classes(ax, X, y, title, alpha=0.75):
    for label, marker, name in CLASS_STYLES:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], marker=marker, s=42, edgecolor="black", linewidth=0.4, alpha=alpha, label=name)
    ax.set(xlabel="feature x1", ylabel="feature x2", title=title)
    ax.legend()


def annotate_probes(ax, coordinates):
    for name, point in zip(probe_names, coordinates):
        ax.scatter(*point, s=170, facecolors="none", edgecolors="black", linewidths=1.8)
        ax.annotate(name, point, xytext=(5, 5), textcoords="offset points", fontsize=8, weight="bold")


fig, ax = plt.subplots(figsize=(9, 5.8))
scatter_classes(ax, X_evidence, y_evidence, "Observe: held-out raw geometry and six probes")
annotate_probes(ax, X_probes)
plt.show()

print(f"{'probe':<14} {'x1':>8} {'x2':>8} {'target':>8}")
for name, point, target in zip(probe_names, X_probes, y_probes):
    print(f"{name:<14} {point[0]:>8.3f} {point[1]:>8.3f} {target:>8d}")

## Predict Before the Fixed-Parameter Reveal

From raw geometry alone, choose exactly two named probes you think the fixed network will classify incorrectly. Predict hidden and output shapes for a batch of 90 examples. State whether you expect a two-dimensional hidden view to clarify every point or only most points, and name evidence that could make you revise your story.

In [ ]:
likely_failure_probes = []  # TODO: enter exactly two names from probe_names.
forward_predictions = {
    "hidden_shape_for_90": "",
    "output_shape_for_90": "",
    "hidden_view_expectation": "",
    "revision_evidence": "",
}
assert len(likely_failure_probes) == 2 and len(set(likely_failure_probes)) == 2, (
    "Prediction checkpoint: choose exactly two distinct probe names before continuing."
)
assert set(likely_failure_probes).issubset(set(probe_names))
assert all(value.strip() for value in forward_predictions.values()), (
    "Prediction checkpoint: complete every forward-shape and evidence field."
)

## Observe: Fixed `2 -> 6 -> 1` Parameters

These values were fitted offline on the supplied training split and are fixed for this investigation. The hidden activation is tanh and the binary output uses sigmoid with threshold `0.5`. Keep the canonical dictionary unchanged.

In [ ]:
parameters = {
    "W1": np.array([
        [-0.16618268, -0.65779348, 3.31297856, -4.42433680, 2.39102480, 5.51100765],
        [-4.92828506, 4.17901840, -2.65367289, -1.22690598, 5.44204936, -0.31550093],
    ]),
    "b1": np.array([4.15543050, 1.15426212, -1.37874150, -1.76356714, -1.36201945, -7.98641624]),
    "W2": np.array([[4.42064156], [-3.64026782], [-3.12646121], [-6.86956055], [-2.85291978], [5.85951811]]),
    "b2": np.array([-0.79449932]),
}
parameter_snapshot = {name: values.copy() for name, values in parameters.items()}
assert parameters['W1'].shape == (2, 6) and parameters['b1'].shape == (6,)
assert parameters['W2'].shape == (6, 1) and parameters['b2'].shape == (1,)
print("Fixed parameter shapes:", {name: values.shape for name, values in parameters.items()})

## Modify: Complete the Vectorized Forward Pass

Complete only the marked TODO bodies. Use batch-first matrix operations; do not loop over examples. `batch_forward` must return a cache with `X`, `Z1`, `A1`, `Z2`, and `P`, where `P` has shape `(n, 1)`.

In [ ]:
def stable_sigmoid(values):
    # TODO: preserve shape and remain finite at extreme magnitudes.
    raise NotImplementedError("TODO: implement a numerically stable sigmoid")


def hidden_step(X, parameters):
    # TODO: compute Z1 = X @ W1 + b1 and A1 = tanh(Z1); return both.
    raise NotImplementedError("TODO: implement the vectorized hidden step")


def output_step(A1, parameters):
    # TODO: compute Z2 = A1 @ W2 + b2 and P = stable_sigmoid(Z2); return both.
    raise NotImplementedError("TODO: implement the vectorized output step")


def batch_forward(X, parameters):
    # TODO: call both steps and return the required cache dictionary.
    raise NotImplementedError("TODO: assemble the complete forward cache")


def predict_classes(probabilities, threshold=0.5):
    # TODO: convert `(n, 1)` probabilities to integer shape `(n,)`.
    raise NotImplementedError("TODO: threshold probabilities into class predictions")

In [ ]:
evidence_cache = batch_forward(X_evidence, parameters)
evidence_predictions = predict_classes(evidence_cache['P'])
evidence_accuracy = np.mean(evidence_predictions == y_evidence)
probe_cache = batch_forward(X_probes, parameters)
probe_predictions = predict_classes(probe_cache['P'])
probe_correct = probe_predictions == y_probes

assert evidence_cache['Z1'].shape == evidence_cache['A1'].shape == (90, 6)
assert evidence_cache['Z2'].shape == evidence_cache['P'].shape == (90, 1)
assert np.all(np.isfinite(evidence_cache['P']))
assert np.all((0.0 <= evidence_cache['P']) & (evidence_cache['P'] <= 1.0))
assert evidence_predictions.shape == (90,)
print(f"Held-out accuracy: {evidence_accuracy:.4f}")
print("Evidence shapes:", {name: value.shape for name, value in evidence_cache.items()})

## Run: Six-Probe Evidence Table

Compare this generated evidence with your raw-geometry prediction. Do not stop at correct/incorrect: inspect probability distance from the threshold and the hidden/output ranges for each trace.

In [ ]:
print(f"{'probe':<14} {'target':>7} {'z2':>9} {'p':>9} {'pred':>7} {'result':>9}")
for row, name in enumerate(probe_names):
    result = "correct" if probe_correct[row] else "wrong"
    print(f"{name:<14} {y_probes[row]:>7d} {probe_cache['Z2'][row, 0]:>9.3f} {probe_cache['P'][row, 0]:>9.3f} {probe_predictions[row]:>7d} {result:>9}")
print(f"Probe correctness: {np.sum(probe_correct)}/6")
assert np.sum(probe_correct) == 4 and np.sum(~probe_correct) == 2

## Predict and Diagnose: Deliberate Transpose Failure

A faulty hidden step uses `X @ W1.T`. Before running it, write the two operand shapes, the inner dimensions NumPy tries to align, and whether this is an implementation failure or a complete-but-wrong model output.

In [ ]:
transpose_prediction = {"operand_shapes": "", "inner_dimensions": "", "failure_type": ""}
assert all(value.strip() for value in transpose_prediction.values()), (
    "Prediction checkpoint: record all transpose-shape predictions before running the defect."
)

In [ ]:
def faulty_transposed_hidden(X, parameters):
    return np.tanh(X @ parameters['W1'].T + parameters['b1'])


transpose_failure_observed = False
try:
    faulty_transposed_hidden(X_probes, parameters)
except ValueError as error:
    transpose_failure_observed = True
    print("Expected deliberate shape failure:", error)

recovered_Z1, recovered_A1 = hidden_step(X_probes, parameters)
assert transpose_failure_observed
assert recovered_Z1.shape == recovered_A1.shape == (6, 6)
print("Recovery shape from the batch-first hidden step:", recovered_A1.shape)

## Modify: Prepare Boundary and Hidden Evidence

Complete the two assignment TODOs. Run `batch_forward` over the supplied mesh. Then call the supplied hidden projection helper with the held-out and probe hidden activations. The projection uses one output-aligned axis and one principal context axis; it is descriptive and cannot preserve every relationship in six dimensions.

In [ ]:
def make_mesh(X, padding=0.45, resolution=180):
    x1 = np.linspace(X[:, 0].min() - padding, X[:, 0].max() + padding, resolution)
    x2 = np.linspace(X[:, 1].min() - padding, X[:, 1].max() + padding, resolution)
    xx1, xx2 = np.meshgrid(x1, x2)
    return xx1, xx2, np.column_stack([xx1.ravel(), xx2.ravel()])


def hidden_projection(reference_hidden, probe_hidden, output_weights):
    centered = reference_hidden - reference_hidden.mean(axis=0, keepdims=True)
    _, _, components = np.linalg.svd(centered, full_matrices=False)
    context_direction = components[0]
    reference_projection = np.column_stack([
        (reference_hidden @ output_weights).ravel(),
        centered @ context_direction,
    ])
    probe_projection = np.column_stack([
        (probe_hidden @ output_weights).ravel(),
        (probe_hidden - reference_hidden.mean(axis=0, keepdims=True)) @ context_direction,
    ])
    return reference_projection, probe_projection


xx1, xx2, mesh_X = make_mesh(X_evidence)
mesh_cache = None  # TODO: run batch_forward on mesh_X with canonical parameters.
hidden_coordinates = None  # TODO: call hidden_projection for held-out and probe A1 values.
assert mesh_cache is not None and hidden_coordinates is not None, (
    "Visualization checkpoint: replace both None values with the requested function calls."
)
evidence_hidden_2d, probe_hidden_2d = hidden_coordinates
mesh_predictions = predict_classes(mesh_cache['P']).reshape(xx1.shape)
assert mesh_cache['P'].shape == (len(mesh_X), 1)
assert evidence_hidden_2d.shape == (90, 2) and probe_hidden_2d.shape == (6, 2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scatter_classes(axes[0, 0], X_evidence, y_evidence, "A. Raw held-out data and probes")
annotate_probes(axes[0, 0], X_probes)

axes[0, 1].contourf(xx1, xx2, mesh_predictions, levels=[-0.5, 0.5, 1.5], colors=["#d9e6f2", "#f3ddbd"], alpha=0.7)
axes[0, 1].contour(xx1, xx2, mesh_cache['P'].reshape(xx1.shape), levels=[0.5], colors="black", linewidths=2)
scatter_classes(axes[0, 1], X_evidence, y_evidence, "B. Fixed-parameter decision region")
annotate_probes(axes[0, 1], X_probes)

scatter_classes(axes[1, 0], evidence_hidden_2d, y_evidence, "C. Two-dimensional hidden projection")
annotate_probes(axes[1, 0], probe_hidden_2d)
axes[1, 0].axvline(-parameters['b2'][0], color="black", linewidth=2, label="output threshold")
axes[1, 0].set(xlabel="hidden output contribution", ylabel="principal context coordinate")
axes[1, 0].legend()

axes[1, 1].axis('off')
trace_lines = ["D. Six-probe forward trace", "", "name          z1 range       a1 range       z2       p   result"]
for row, name in enumerate(probe_names):
    result = "correct" if probe_correct[row] else "wrong"
    trace_lines.append(
        f"{name:<13} [{probe_cache['Z1'][row].min():>5.2f},{probe_cache['Z1'][row].max():>5.2f}] "
        f"[{probe_cache['A1'][row].min():>5.2f},{probe_cache['A1'][row].max():>5.2f}] "
        f"{probe_cache['Z2'][row, 0]:>6.2f} {probe_cache['P'][row, 0]:>6.3f} {result}"
    )
axes[1, 1].text(0.02, 0.96, '\n'.join(trace_lines), va='top', family='monospace', fontsize=9)

plt.tight_layout()
plt.show()

## Diagnose the Mystery

For one correct and one incorrect probe, trace raw location -> hidden range/projection -> output logit -> sigmoid output -> thresholded class. State which evidence changed or strengthened your original prediction.

Classify each event as an implementation/interface failure, a semantic-invariant failure, or a complete fixed-parameter prediction that disagrees with the target. State what this forward evidence cannot reveal about how the parameters were obtained.

In [ ]:
mystery_diagnosis = {
    "one_correct_trace": "",
    "one_incorrect_trace": "",
    "prediction_revision": "",
    "transpose_failure_type": "",
    "wrong_probe_failure_type": "",
    "forward_evidence_limit": "",
}
assert all(value.strip() for value in mystery_diagnosis.values()), (
    "Diagnosis checkpoint: complete all six evidence statements before perturbing parameters."
)

## Extend: One Copied-Parameter Perturbation

Choose exactly one element from `W1`, `b1`, `W2`, or `b2` and a nonzero delta. Before running, predict a local effect on at least one probe probability or boundary region. The experiment operates on a deep copy; the canonical parameters must remain unchanged.

In [ ]:
perturbation = {
    "parameter": "",  # TODO: W1, b1, W2, or b2.
    "index": None,  # TODO: tuple such as (0, 0) or (0,).
    "delta": None,  # TODO: nonzero scalar with magnitude at least 0.05.
    "predicted_local_effect": "",
}
assert perturbation['parameter'] in parameters
assert isinstance(perturbation['index'], tuple)
assert perturbation['delta'] is not None and abs(float(perturbation['delta'])) >= 0.05
assert perturbation['predicted_local_effect'].strip()
assert len(perturbation['index']) == parameters[perturbation['parameter']].ndim

perturbed_parameters = copy.deepcopy(parameters)
perturbed_parameters[perturbation['parameter']][perturbation['index']] += float(perturbation['delta'])

In [ ]:
perturbed_evidence_cache = batch_forward(X_evidence, perturbed_parameters)
perturbed_probe_cache = batch_forward(X_probes, perturbed_parameters)
perturbed_predictions = predict_classes(perturbed_evidence_cache['P'])
perturbed_probe_predictions = predict_classes(perturbed_probe_cache['P'])
probability_shift = perturbed_probe_cache['P'].ravel() - probe_cache['P'].ravel()

print(f"Changed {perturbation['parameter']}{perturbation['index']} by {float(perturbation['delta']):+.3f}")
print(f"Held-out accuracy: baseline={evidence_accuracy:.4f}, perturbed={np.mean(perturbed_predictions == y_evidence):.4f}")
for name, shift, old_class, new_class in zip(probe_names, probability_shift, probe_predictions, perturbed_probe_predictions):
    print(f"{name:<14} probability_shift={shift:+.4f} class {old_class} -> {new_class}")
assert any(not np.array_equal(parameters[name], perturbed_parameters[name]) for name in parameters)
assert all(np.array_equal(parameters[name], parameter_snapshot[name]) for name in parameters)
assert not np.allclose(perturbed_probe_cache['P'], probe_cache['P'])

## Explain and Reflect

Complete the final evidence record:

1. Did the perturbation support your local prediction? Cite a probability or class change.
2. Why do fixed parameters, rather than an inference-time learning process, produce every result in this notebook?
3. What would Day 2 need to add for an incorrect example to influence future parameters?
4. Why is the two-dimensional hidden projection useful but incomplete?

In [ ]:
reflection = {
    "perturbation_result": "",
    "fixed_parameter_mechanism": "",
    "day2_missing_piece": "",
    "projection_limit": "",
}
assert all(value.strip() for value in reflection.values()), "Reflection checkpoint: complete all four evidence statements."
assert 0.82 <= evidence_accuracy <= 0.92
assert evidence_cache['A1'].shape == (90, 6) and evidence_cache['P'].shape == (90, 1)
assert np.sum(probe_correct) == 4 and np.sum(~probe_correct) == 2
assert transpose_failure_observed
assert np.array_equal(predict_classes(batch_forward(mesh_X, parameters)['P']), mesh_predictions.ravel())
assert all(np.array_equal(parameters[name], parameter_snapshot[name]) for name in parameters)
print("LAB-D1-04 checkpoint passed: held-out band, shapes/ranges, 4/2 probes, mesh parity, transpose recovery, and parameter immutability.")

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Prediction checkpoint stops | Two names or a shape/evidence response is missing | Complete the commitment before revealing parameters |
| Hidden matrix multiplication fails | `W1` was transposed or examples are not rows | Restore `(n,2) @ (2,6)` |
| Output is `(n,)` instead of `(n,1)` | A singleton dimension was squeezed too early | Keep `W2` `(6,1)` and `b2` `(1,)` until class conversion |
| Probabilities overflow | Sigmoid used an unstable direct exponent | Branch by sign or use an equivalent stable formula |
| Probe behavior differs | Seed, split, probe indices, or fixed values changed | Restart and rerun supplied setup unchanged |
| Visualization checkpoint stops | Mesh or hidden assignments remain `None` | Call the named helpers with canonical parameters/cache values |
| Perturbation mutates baseline | The canonical dictionary was edited | Restart and use `copy.deepcopy(parameters)` |

## Optional Extension

Repeat the perturbation with the same parameter element and the opposite delta. Predict whether probability shifts will be symmetric; explain any observed asymmetry using tanh or sigmoid nonlinearity. Keep this outside the required one-experiment record.

## Takeaways

- A vectorized `2 -> 6 -> 1` forward pass produces hidden shape `(n,6)` and output shape `(n,1)`.
- Raw, boundary, hidden, and trace evidence answer different questions about one fixed computation.
- A transpose mismatch is an interface failure; a completed wrong class is a fixed-parameter prediction error.
- A two-dimensional hidden projection is descriptive evidence, not a complete account of six dimensions.
- Inference traces outputs; training must add an objective and a mechanism for changing parameters.

Return to the [LAB-D1-04 debrief](../student-guide/day-1-student-guide.md#lab-d1-04---build-a-network-that-thinks-forward). Review [LESSON-D1-06](../student-guide/day-1-student-guide.md#lesson-d1-06---forward-propagation-with-numpy), [LAB-D1-02](LAB-D1-02-linear-limit.ipynb), and [LAB-D1-03](LAB-D1-03-shapes-activations.ipynb) as needed.